**06 · Verificación de resultados y checklist de cierre**

Ejecutar desde la raíz de `point-field`. Este cuaderno no reemplaza la revisión manual, pero automatiza la verificación de archivos, tablas y cifras clave antes de la última pulida del documento.

In [2]:
from pathlib import Path
import sys, subprocess, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'benchmarks'/'common.py').exists() and (p/'pyproject.toml').exists()), None)
if ROOT is None:
    raise RuntimeError("Abre Jupyter desde la raíz de point-field o una subcarpeta del repositorio.")
sys.path.insert(0, str(ROOT/'benchmarks'))
OUT = ROOT/'runs'/'tesis_figuras'
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'figure.dpi': 120, 'savefig.dpi': 300, 'pdf.fonttype': 42})
COLORS = ['#21618c','#c65036','#63933e','#8b65a5','#c49a38','#458e91','#d16c9f','#555555']
def save(fig, name):
    for ext in ('pdf','png'):
        fig.savefig(OUT/f'{name}.{ext}', bbox_inches='tight', facecolor='white')
    plt.show(); plt.close(fig)
def points(ax, X, y=None, s=6):
    X=np.asarray(X)
    if y is None:
        if X.shape[1]==2: ax.scatter(X[:,0],X[:,1],s=s,c=COLORS[0],linewidths=0)
        else: ax.scatter(X[:,0],X[:,1],X[:,2],s=s,c=COLORS[0],linewidths=0)
    else:
        labs=np.unique(y)
        for j,c in enumerate(labs):
            col='#b9b9b9' if c==-1 else COLORS[j%len(COLORS)]
            m=(y==c)
            if X.shape[1]==2: ax.scatter(X[m,0],X[m,1],s=s,c=col,linewidths=0)
            else: ax.scatter(X[m,0],X[m,1],X[m,2],s=s,c=col,linewidths=0)
    ax.set_xticks([]); ax.set_yticks([])
    if X.shape[1]==2:
        ax.set_aspect('equal', adjustable='box')
    else:
        ax.set_zticks([]); ax.set_box_aspect((1,1,1)); ax.view_init(elev=23, azim=35)
def run_script(name, *args):
    subprocess.run([sys.executable, str(ROOT/'benchmarks'/name), *map(str,args)], cwd=ROOT, check=True)
print('Repositorio:', ROOT, 'Salidas:', OUT)
try:
    commit=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip()
except Exception:
    commit='No disponible'
print('Commit:', commit)


Repositorio: /home/icantu24/Documents/PEF-CD/point-field Salidas: /home/icantu24/Documents/PEF-CD/point-field/runs/tesis_figuras
Commit: 88445aa61ca2a66afcc53a931cf0927232ef232f


**1. Manifesto mínimo de archivos.** Ajusta la lista si tus archivos históricos tienen nombres diferentes. El objetivo es saber qué existe, qué falta y qué se va a volver a generar.

In [3]:
EXPECTED = {
    'clustering_2d': [ROOT/'benchmarks/results/clustering_2d_eval.csv'],
    'objetos_sinteticos': [ROOT/'benchmarks/results/objetos_3d.csv'],
    'modelnet10': [ROOT/'benchmarks/results/modelnet10_test.csv'],
    'modelnet40': [ROOT/'benchmarks/results/modelnet40_test.csv'],
    'embeddings': [ROOT/'benchmarks/results/embedding_2d.csv', ROOT/'benchmarks/results/embeddings_2d.csv'],
    'contacto3d': [ROOT/'benchmarks/results/contacto_3d.csv', ROOT/'benchmarks/results/contact_3d.csv'],
    'flare': [ROOT/'benchmarks/results/flare_test.csv', ROOT/'benchmarks/results/flare.csv'],
}
rows=[]
for key,candidates in EXPECTED.items():
    found = next((p for p in candidates if p.exists()), None)
    rows.append({'bloque': key, 'archivo': str(found) if found else 'NO ENCONTRADO', 'estado': 'OK' if found else 'FALTA'})
manifest = pd.DataFrame(rows)
display(manifest)
manifest.to_csv(OUT/'06_manifest_resultados.csv', index=False)


,bloque,archivo,estado
0,clustering_2d,/home/icantu24/Documents/PEF-CD/point-field/be...,OK
1,objetos_sinteticos,/home/icantu24/Documents/PEF-CD/point-field/be...,OK
2,modelnet10,/home/icantu24/Documents/PEF-CD/point-field/be...,OK
3,modelnet40,/home/icantu24/Documents/PEF-CD/point-field/be...,OK
4,embeddings,NO ENCONTRADO,FALTA
5,contacto3d,NO ENCONTRADO,FALTA
6,flare,NO ENCONTRADO,FALTA


**2. Resúmenes automáticos.** Si un CSV existe, se resume con medias, desviaciones y tablas pivote. Esto te permite cotejar si las cifras del manuscrito coinciden con lo que realmente está en disco.

In [4]:
def first_existing(paths):
    return next((p for p in paths if p.exists()), None)

def summarize_csv(path):
    df=pd.read_csv(path)
    print('Archivo:', path)
    print('Columnas:', list(df.columns))
    display(df.head())
    numeric=df.select_dtypes(include='number')
    if not numeric.empty:
        display(numeric.describe().T)
    return df

loaded={}
for key,paths in EXPECTED.items():
    path=first_existing(paths)
    if path:
        try:
            loaded[key]=summarize_csv(path)
        except Exception as e:
            print('No se pudo resumir', key, e)


Archivo: /home/icantu24/Documents/PEF-CD/point-field/benchmarks/results/clustering_2d_eval.csv
Columnas: ['case', 'held_out', 'variant', 'seed', 'method', 'family', 'receives_k', 'n_points', 'ARI', 'NMI', 'n_found', 'noise_frac', 'time_s', 'szego_active', 'winding_gain', 'dbscan_eps']


,case,held_out,variant,seed,method,family,receives_k,n_points,ARI,NMI,n_found,noise_frac,time_s,szego_active,winding_gain,dbscan_eps
0,compacto,False,base,100,K-Means,referencia,True,450,0.903218,0.869259,3,0.000000,0.801771,NaN,NaN,NaN
1,compacto,False,base,100,DBSCAN,referencia,False,450,0.858343,0.809593,3,0.060000,0.003726,NaN,NaN,0.16747
2,compacto,False,base,100,OPTICS,referencia,False,450,0.487982,0.579457,3,0.355556,0.252167,NaN,NaN,NaN
3,compacto,False,base,100,Spectral (kNN),referencia,True,450,0.903458,0.875769,3,0.000000,0.636780,NaN,NaN,NaN
4,compacto,False,base,100,CD global (inicial),CD,True,450,0.915759,0.887091,3,0.000000,0.786937,NaN,NaN,NaN


,count,mean,std,min,25%,50%,75%,max
seed,840.0,102.000000,1.415056,100.000000,101.000000,102.000000,103.000000,104.000000
n_points,840.0,291.783333,86.959120,155.000000,225.000000,300.000000,309.250000,450.000000
ARI,840.0,0.422130,0.378854,-0.007366,0.096837,0.253812,0.903659,1.000000
NMI,840.0,0.469153,0.351681,0.000000,0.149595,0.395008,0.864025,1.000000
n_found,840.0,3.548810,3.091377,1.000000,2.000000,3.000000,3.000000,22.000000
noise_frac,840.0,0.064924,0.150143,0.000000,0.000000,0.000000,0.027827,0.791111
time_s,840.0,0.285825,0.577636,0.001858,0.008243,0.033126,0.218630,4.610847
winding_gain,120.0,0.133996,0.108408,0.000285,0.038512,0.108565,0.206081,0.436892
dbscan_eps,120.0,0.171553,0.039511,0.103138,0.145755,0.164706,0.183691,0.294646


Archivo: /home/icantu24/Documents/PEF-CD/point-field/benchmarks/results/objetos_3d.csv
Columnas: ['condition', 'method', 'mAP', 'nn', 'ci_low', 'ci_high']


,condition,method,mAP,nn,ci_low,ci_high
0,clean,CD_U,0.912464,1.000000,NaN,NaN
1,clean,CD_W,0.915957,1.000000,NaN,NaN
2,clean,M_U,0.840234,1.000000,NaN,NaN
3,clean,D2,0.758929,0.957143,NaN,NaN
4,clean,CD_U_metric,0.939663,1.000000,NaN,NaN


,count,mean,std,min,25%,50%,75%,max
mAP,40.0,0.554717,0.346660,-0.293310,0.330966,0.663490,0.870140,0.939663
nn,30.0,0.796667,0.218100,0.342857,0.592857,0.892857,1.000000,1.000000
ci_low,10.0,0.006484,0.127961,-0.332304,0.001693,0.032548,0.037477,0.149623
ci_high,10.0,0.052796,0.130019,-0.253674,0.018927,0.067023,0.078425,0.217133


... (2 outputs left out)

,count,mean,std,min,25%,50%,75%,max
mAP,45.0,0.257294,0.150024,-0.077181,0.203751,0.321895,0.372289,0.406359
nn,35.0,0.549571,0.173859,0.165000,0.425000,0.615000,0.695000,0.760000
ci_low,10.0,-0.006806,0.034134,-0.088602,-0.016003,-0.001880,0.010184,0.044224
ci_high,10.0,0.014297,0.034327,-0.065620,0.005386,0.016522,0.030511,0.069873


Archivo: /home/icantu24/Documents/PEF-CD/point-field/benchmarks/results/modelnet40_test.csv
Columnas: ['condition', 'method', 'mAP', 'nn', 'ci_low', 'ci_high']


,condition,method,mAP,nn,ci_low,ci_high
0,clean,CD_U,0.224984,0.4750,NaN,NaN
1,clean,CD_W,0.215241,0.4625,NaN,NaN
2,clean,M_U,0.220037,0.4800,NaN,NaN
3,clean,D2,0.151321,0.3225,NaN,NaN
4,clean,CD_U_metric,0.244542,0.5225,NaN,NaN


,count,mean,std,min,25%,50%,75%,max
mAP,45.0,0.140417,0.085703,-0.051975,0.082353,0.153498,0.212818,0.246382
nn,35.0,0.357143,0.138812,0.092500,0.262500,0.357500,0.468750,0.552500
ci_low,10.0,-0.000980,0.026410,-0.058596,-0.003779,-0.001641,0.003373,0.051032
ci_high,10.0,0.011664,0.027176,-0.045659,0.007264,0.011033,0.014707,0.066790


**3. Verificación de cifras clave reclamadas en la tesis.** Los alias y los nombres de columna pueden variar entre versiones; ajusta el diccionario si tu CSV usa nombres diferentes. El resultado es una tabla PASS/WARN con tolerancia configurable.

In [5]:
ALIASES = {
    'V0': ['V0', 'CD global', 'global'],
    'V1': ['V1', 'CD local', 'local'],
    'V2': ['V2', 'fase', 'phase'],
    'ESPECTRAL': ['Spectral', 'Espectral', 'knn'],
    'CD_M': ['CD + m', 'CD+m', 'CD_U+m', 'CD metric'],
    'MOM_M': ['Momentos + m', 'M_U+m', 'moment metric', 'momentos+m'],
    'CD': ['CD', 'CD_U'],
    'MOM': ['Momentos', 'M_U'],
}

CLAIMS = [
    {'bloque':'modelnet10','condition':'clean','method':'CD_M','metric':'mAP','expected':0.400,'tol':0.03},
    {'bloque':'modelnet10','condition':'clean','method':'MOM_M','metric':'mAP','expected':0.406,'tol':0.03},
    {'bloque':'modelnet40','condition':'clean','method':'CD_M','metric':'mAP','expected':0.245,'tol':0.03},
    {'bloque':'modelnet40','condition':'clean','method':'MOM_M','metric':'mAP','expected':0.236,'tol':0.03},
    {'bloque':'flare','condition':'all','method':'CD_M','metric':'mAP','expected':0.326,'tol':0.03},
    {'bloque':'flare','condition':'all','method':'MOM_M','metric':'mAP','expected':0.282,'tol':0.03},
]

def find_col(df, names):
    low={c.lower():c for c in df.columns}
    for n in names:
        if n.lower() in low:
            return low[n.lower()]
    for c in df.columns:
        if any(n.lower() in c.lower() for n in names):
            return c
    return None

checks=[]
for claim in CLAIMS:
    df=loaded.get(claim['bloque'])
    if df is None:
        checks.append({**claim, 'observed':np.nan, 'status':'MISSING_FILE'})
        continue
    mcol=find_col(df,['method'])
    ccol=find_col(df,['condition','variant'])
    vcol=find_col(df,[claim['metric']])
    if mcol is None or vcol is None:
        checks.append({**claim,'observed':np.nan,'status':'MISSING_COLUMN'})
        continue
    sub=df.copy()
    pats=ALIASES.get(claim['method'], [claim['method']])
    sub=sub[sub[mcol].astype(str).str.contains('|'.join(pats), case=False, regex=True)]
    cond=claim['condition']
    if ccol is not None and cond != 'all':
        sub=sub[sub[ccol].astype(str).str.contains(cond, case=False, regex=False)]
    observed = sub[vcol].mean() if len(sub) else np.nan
    ok = np.isfinite(observed) and abs(observed-claim['expected'])<=claim['tol']
    checks.append({**claim,'observed':observed,'status':'PASS' if ok else ('NO_MATCH' if len(sub)==0 else 'WARN')})
check_df=pd.DataFrame(checks)
display(check_df)
check_df.to_csv(OUT/'06_checks_tesis.csv', index=False)


,bloque,condition,method,metric,expected,tol,observed,status
0,modelnet10,clean,CD_M,mAP,0.400,0.03,NaN,NO_MATCH
1,modelnet10,clean,MOM_M,mAP,0.406,0.03,NaN,NO_MATCH
2,modelnet40,clean,CD_M,mAP,0.245,0.03,NaN,NO_MATCH
3,modelnet40,clean,MOM_M,mAP,0.236,0.03,NaN,NO_MATCH
4,flare,all,CD_M,mAP,0.326,0.03,NaN,MISSING_FILE
5,flare,all,MOM_M,mAP,0.282,0.03,NaN,MISSING_FILE


**4. Checklist manual final.** Esta tabla se llena automáticamente con los bloques principales para que tú marques qué ya quedó validado, qué se va a volver a correr y qué quedará como limitación declarada.

In [6]:
checklist = pd.DataFrame([
    ['Soporte 2D','Figura de grado y puntos',''],
    ['Soporte 3D','Cobertura y franja próxima',''],
    ['Clustering 2D','Panel V0–V2, heatmaps y robustez',''],
    ['Clustering 3D','Escenas separadas y en contacto',''],
    ['Embeddings','Trustworthiness, continuity, 5-NN',''],
    ['Objetos sintéticos','mAP, 1-NN, invariancia/robustez',''],
    ['ModelNet10','mAP, 1-NN, comparación DNN',''],
    ['ModelNet40','mAP, 1-NN, comparación DNN',''],
    ['FLARE','mAP, 1-NN, alcance exploratorio',''],
    ['Documento','Citas, tablas, figuras y números consistentes',''],
], columns=['bloque','qué revisar','estado_manual'])
display(checklist)
checklist.to_csv(OUT/'06_checklist_manual.csv', index=False)
print('Guardados: 06_manifest_resultados.csv, 06_checks_tesis.csv y 06_checklist_manual.csv')


,bloque,qué revisar,estado_manual
0,Soporte 2D,Figura de grado y puntos,
1,Soporte 3D,Cobertura y franja próxima,
2,Clustering 2D,"Panel V0–V2, heatmaps y robustez",
3,Clustering 3D,Escenas separadas y en contacto,
4,Embeddings,"Trustworthiness, continuity, 5-NN",
5,Objetos sintéticos,"mAP, 1-NN, invariancia/robustez",
6,ModelNet10,"mAP, 1-NN, comparación DNN",
7,ModelNet40,"mAP, 1-NN, comparación DNN",
8,FLARE,"mAP, 1-NN, alcance exploratorio",
9,Documento,"Citas, tablas, figuras y números consistentes",


Guardados: 06_manifest_resultados.csv, 06_checks_tesis.csv y 06_checklist_manual.csv
